In [13]:
import heapq
import copy

class PuzzleState:
    def __init__(self, board, goal, g=0, parent=None):
        self.board = board
        self.n = len(board)
        self.goal = goal
        self.g = g
        self.parent = parent
        self.h = self.calculate_manhattan()
        self.f = self.g + self.h

    def calculate_manhattan(self):
        distance = 0
        goal_map = {}
        for r in range(self.n):
            for c in range(self.n):
                goal_map[self.goal[r][c]] = (r, c)

        for r in range(self.n):
            for c in range(self.n):
                val = self.board[r][c]
                if val != 0:
                    target_r, target_c = goal_map[val]
                    distance += abs(r - target_r) + abs(c - target_c)
        return distance

    def get_neighbors(self):
        neighbors = []
        r, c = next((r, c) for r in range(self.n) for c in range(self.n) if self.board[r][c] == 0)
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.n and 0 <= nc < self.n:
                new_board = [row[:] for row in self.board]
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                neighbors.append(PuzzleState(new_board, self.goal, self.g + 1, self))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

    def get_path(self):
        path = []
        curr = self
        while curr:
            path.append(curr.board)
            curr = curr.parent
        return path[::-1]

def solve_n_puzzle(start_board, goal_board):
    start_state = PuzzleState(start_board, goal_board)
    open_list = [start_state]
    closed_set = set()

    while open_list:
        current = heapq.heappop(open_list)

        if current.board == goal_board:
            return current.get_path(), current.g

        board_tuple = tuple(tuple(row) for row in current.board)
        if board_tuple in closed_set:
            continue
        closed_set.add(board_tuple)

        for neighbor in current.get_neighbors():
            heapq.heappush(open_list, neighbor)

    return None, 0

In [14]:
# Ví dụ bài toán 15-puzzle (4x4)
# 0 đại diện cho ô trống
start_15 = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 0, 11],
    [13, 14, 15, 12]
]

goal_15 = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

path, steps = solve_n_puzzle(start_15, goal_15)

if path:
    print(f"Đã tìm thấy lời giải sau {steps} bước di chuyển.")
    print("Trạng thái cuối cùng:")
    for row in path[-1]:
        print(row)
else:
    print("Không tìm thấy lời giải.")

Đã tìm thấy lời giải sau 2 bước di chuyển.
Trạng thái cuối cùng:
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [15]:

#Cài đặt bài toán người giao hàng theo thuật giải A*.import heapq
def get_mst_weight(nodes, distance_matrix):
    if not nodes:
        return 0
    nodes = list(nodes)
    unvisited = set(nodes)
    if not unvisited:
        return 0
    start_node = nodes[0]
    unvisited.remove(start_node)
    edges = []
    for n in unvisited:
        heapq.heappush(edges, (distance_matrix[start_node][n], n))

    mst_weight = 0
    while unvisited and edges:
        weight, u = heapq.heappop(edges)
        if u in unvisited:
            unvisited.remove(u)
            mst_weight += weight
            for v in unvisited:
                heapq.heappush(edges, (distance_matrix[u][v], v))
    return mst_weight

def tsp_a_star(distance_matrix):
    n = len(distance_matrix)
    start_node = 0
    all_visited = (1 << n) - 1

    # Priority Queue: (f_score, current_node, visited_mask, current_cost, path)
    pq = [(0, start_node, 1 << start_node, 0, [start_node])]

    while pq:
        f, u, mask, g, path = heapq.heappop(pq)

        if mask == all_visited:
            # Return to start node
            final_cost = g + distance_matrix[u][start_node]
            return path + [start_node], final_cost

        unvisited_nodes = [v for v in range(n) if not (mask & (1 << v))]

        for v in unvisited_nodes:
            new_g = g + distance_matrix[u][v]
            # Heuristic: MST of unvisited nodes + dist to nearest unvisited + unvisited to start
            h = get_mst_weight(unvisited_nodes, distance_matrix)
            new_f = new_g + h

            heapq.heappush(pq, (new_f, v, mask | (1 << v), new_g, path + [v]))

    return None, float('inf')

In [16]:
# Ma trận khoảng cách giữa 4 thành phố (0, 1, 2, 3)
distance_matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

path, total_cost = tsp_a_star(distance_matrix)

print(f"Đường đi tối ưu: {' -> '.join(map(str, path))}")
print(f"Tổng chi phí nhỏ nhất: {total_cost}")

Đường đi tối ưu: 0 -> 1 -> 3 -> 2 -> 0
Tổng chi phí nhỏ nhất: 80
